# 02 - Kafka Streaming Verisini Bronze Delta Katmanına Yazma

Bu notebook, MovieLens `ratings.csv` verisinin streaming veri gibi Kafka'ya gönderilmesini ve Spark Structured Streaming ile Delta Bronze katmanına yazılmasını açıklar.

## Amaç

Gerçek dünyada bir film platformunda kullanıcılar sürekli rating üretir. Bu projede bu gerçek zamanlı veri akışı, CSV dosyasından okunan kayıtların Kafka'ya JSON mesajları olarak gönderilmesiyle simüle edilir.

## Veri akışı

`ratings.csv → Python Kafka Producer → Kafka topic: movie-ratings → Spark Structured Streaming → Delta Bronze`

## Kafka Producer kodu

Bu kod `ratings.csv` dosyasını chunk'lar halinde okur. Her satırı JSON formatına çevirir ve Kafka'daki `movie-ratings` topic'ine gönderir.

```bash
import os
import json
import time
import pandas as pd
from kafka import KafkaProducer


KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:29092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "movie-ratings")

RATINGS_PATH = "/app/data/raw/ml-25m/ratings.csv"


def create_producer():
    return KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
        value_serializer=lambda value: json.dumps(value).encode("utf-8"),
        key_serializer=lambda key: str(key).encode("utf-8"),
    )


def main():
    print("Kafka producer başlatılıyor...")
    print(f"Topic: {KAFKA_TOPIC}")
    print(f"Dataset path: {RATINGS_PATH}")

    producer = create_producer()

    chunk_size = 1000

    for chunk in pd.read_csv(RATINGS_PATH, chunksize=chunk_size):
        for _, row in chunk.iterrows():
            event = {
                "userId": int(row["userId"]),
                "movieId": int(row["movieId"]),
                "rating": float(row["rating"]),
                "timestamp": int(row["timestamp"]),
            }

            producer.send(
                KAFKA_TOPIC,
                key=event["userId"],
                value=event
            )

        producer.flush()
        print(f"{chunk_size} rating Kafka'ya gönderildi.")
        time.sleep(1)


if __name__ == "__main__":
    main()
```

## Producer kodunun işleyişi

1. Ortam değişkenlerinden Kafka adresi ve topic adı okunur.
2. `/app/data/raw/ml-25m/ratings.csv` dosyası Pandas ile chunk'lar halinde okunur.
3. Her satırdan `userId`, `movieId`, `rating`, `timestamp` alanları alınır.
4. Bu kayıt JSON mesajına çevrilir.
5. Kafka'ya gönderilir.
6. Her chunk sonunda kaç mesaj gönderildiği loglanır.

## Spark Structured Streaming kodu

Bu kod Kafka topic'ini dinler, JSON mesajlarını parse eder ve Delta Bronze katmanına yazar.

```bash
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType


KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:29092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "movie-ratings")

BRONZE_PATH = "/app/delta/bronze/ratings"


schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True),
])


spark = (
    SparkSession.builder
    .appName("MovieRatingsStreamingToDelta")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

parsed_stream = (
    raw_stream
    .selectExpr("CAST(value AS STRING) as json_value")
    .select(from_json(col("json_value"), schema).alias("data"))
    .select("data.*")
    .withColumn("rating_datetime", from_unixtime(col("timestamp")))
)

query = (
    parsed_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/app/delta/bronze/checkpoints/ratings")
    .start(BRONZE_PATH)
)

print("Spark streaming başladı. Kafka'dan veriler okunuyor ve Delta Bronze katmanına yazılıyor.")

query.awaitTermination()
```

## Spark streaming kodunun işleyişi

1. SparkSession Delta Lake desteğiyle başlatılır.
2. Kafka kaynağına bağlanılır.
3. `value` alanındaki JSON string parse edilir.
4. JSON için şema tanımlanır.
5. Unix timestamp okunabilir tarih alanına dönüştürülür.
6. Veri Delta formatında `/app/delta/bronze/ratings` path'ine append modunda yazılır.
7. Checkpoint kullanılarak streaming süreci güvenli hale getirilir.

## Kafka topic oluşturma

```bash
docker exec -it movie-kafka /opt/kafka/bin/kafka-topics.sh --create --topic movie-ratings --bootstrap-server kafka:29092 --partitions 1 --replication-factor 1
docker exec -it movie-kafka /opt/kafka/bin/kafka-topics.sh --list --bootstrap-server kafka:29092
```

## Spark streaming job'unu başlatma

```bash
docker exec -it movie-spark bash
spark-submit --packages io.delta:delta-spark_2.12:3.2.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 streaming_to_delta.py
```

## Producer'ı çalıştırma

```bash
docker compose up producer
```

## Beklenen çıktı

Producer terminalinde:

`1000 rating Kafka'ya gönderildi.`

Delta klasöründe:

`delta/bronze/ratings/_delta_log` ve `.parquet` dosyaları oluşmalıdır.